# 所有当前 Stanza 中文 dependency parser 在 Cantonese-HK test 上的零样本 LAS

固定 `stanza==1.14.0`，从该版本官方 `resources_1.14.0.json` 自动枚举并去重简体 (`zh-hans`) 与繁体 (`zh-hant`) 的完整 POS+lemma+dependency 配置。

Notebook 会从固定的官方 UD r2.18 URL 下载原始 1,004 句文件，按本项目 manifest 中固定的 100 个原始位置重建当前 test，并验证 SHA-256，无需手动上传数据。

**评测条件：**使用粤语 gold 句界和 gold 分词，模型预测 POS、lemma、HEAD、DEPREL。主指标为 Stanza 附带的官方 CoNLL-2018 LAS；另外报告保留 `DEPREL` 子类型的严格 LAS。这里没有训练或微调。建议在 Colab 选择 GPU runtime。

In [ ]:
%pip -q install stanza==1.14.0


In [ ]:
from pathlib import Path
from google.colab import files
import hashlib, json, os, urllib.request

RAW_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Cantonese-HK/r2.18/yue_hk-ud-test.conllu'
EXPECTED_RAW_SHA256 = 'cbd843a195d0db4cdafbf6fcafb7b7b559afea750411006f4728311e70cc4e2a'
EXPECTED_TEST_SHA256 = '1d7b19ac4c0a75d58a80482413ff9cac63ed18917262c18fa1f92fc0f871c0a0'
TEST_PATH = Path('/content/test.conllu')
TEST_ORIGINAL_POSITIONS = [
    7, 26, 28, 31, 33, 45, 47, 73, 82, 83, 91, 96, 97, 101, 105, 106, 117, 132, 149, 166,
    170, 173, 204, 211, 228, 231, 233, 236, 241, 245, 247, 250, 260, 282, 293, 294, 297, 309,
    313, 361, 369, 373, 385, 389, 392, 395, 401, 409, 411, 452, 457, 484, 489, 495, 542, 577,
    586, 594, 604, 621, 633, 634, 646, 648, 663, 673, 680, 684, 695, 707, 722, 744, 748, 751,
    763, 777, 784, 788, 789, 807, 811, 821, 855, 856, 858, 879, 885, 898, 906, 913, 918, 921,
    944, 952, 959, 969, 983, 988, 991, 995,
]

raw_bytes = urllib.request.urlopen(RAW_URL).read()
raw_sha256 = hashlib.sha256(raw_bytes).hexdigest()
if raw_sha256 != EXPECTED_RAW_SHA256:
    raise RuntimeError(f'官方原始文件 SHA-256 不匹配：{raw_sha256}。拒绝从变化的数据重建 test。')

raw_text = raw_bytes.decode('utf-8')
all_blocks, current = [], []
for line in raw_text.splitlines(keepends=True):
    if line.strip():
        current.append(line)
    elif current:
        all_blocks.append(''.join(current))
        current = []
if current:
    all_blocks.append(''.join(current))
if len(all_blocks) != 1004 or len(TEST_ORIGINAL_POSITIONS) != 100:
    raise RuntimeError(f'句块数量异常：raw={len(all_blocks)}, test positions={len(TEST_ORIGINAL_POSITIONS)}')
selected_blocks = [all_blocks[position - 1] for position in TEST_ORIGINAL_POSITIONS]
test_text = ''.join(block + ('' if block.endswith(('\n', '\r')) else '\n') + '\n' for block in selected_blocks)
TEST_PATH.write_text(test_text, encoding='utf-8')

actual_sha256 = hashlib.sha256(TEST_PATH.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_TEST_SHA256:
    raise RuntimeError(f'test.conllu SHA-256 不匹配：{actual_sha256}。拒绝评测错误版本。')
print('downloaded raw SHA-256:', raw_sha256)
print('rebuilt test file:', TEST_PATH)
print('SHA-256:', actual_sha256)


In [ ]:
import stanza
import torch
from stanza.resources.common import download_resources_json, load_resources_json

STANZA_VERSION = '1.14.0'
MODEL_DIR = Path('/content/stanza_resources_1.14.0')
RESULT_DIR = Path('/content/yue_test_stanza_zh_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

assert stanza.__version__ == STANZA_VERSION, stanza.__version__
download_resources_json(model_dir=str(MODEL_DIR))
resources_path = MODEL_DIR / 'resources.json'
resources = load_resources_json(model_dir=str(MODEL_DIR))
resources_sha256 = hashlib.sha256(resources_path.read_bytes()).hexdigest()
print('Stanza:', stanza.__version__)
print('resources SHA-256:', resources_sha256)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
REQUIRED = ('tokenize', 'pos', 'lemma', 'depparse')
LANGS = ('zh-hans', 'zh-hant')

def enumerate_unique_chinese_configs(resources):
    configs = []
    for lang in LANGS:
        lang_resources = resources[lang]
        by_signature = {}
        for alias, mapping in sorted(lang_resources['packages'].items()):
            if not all(processor in mapping for processor in REQUIRED):
                continue
            processor_map = {processor: mapping[processor] for processor in REQUIRED}
            signature = tuple(processor_map[p] for p in REQUIRED)
            by_signature.setdefault(signature, {'processors': processor_map, 'aliases': []})['aliases'].append(alias)
        for signature, item in sorted(by_signature.items()):
            item['lang'] = lang
            item['lang_name'] = lang_resources['lang_name']
            item['config_id'] = f"{lang}__{item['processors']['depparse']}"
            configs.append(item)
    return configs

configs = enumerate_unique_chinese_configs(resources)
print(json.dumps(configs, ensure_ascii=False, indent=2))
assert len(configs) == 4, f'官方 resources 已变化：预期 4 个去重配置，实际 {len(configs)}。请先审阅上方清单。'
assert {c['processors']['depparse'] for c in configs} == {
    'gsdsimp_charlm', 'gsdsimp_nocharlm', 'gsdsimp_electra-large', 'gsd_nocharlm'
}


In [ ]:
def read_conllu(path):
    text = path.read_text(encoding='utf-8')
    blocks = []
    current = []
    for line in text.splitlines():
        if line.strip():
            current.append(line)
        elif current:
            blocks.append(current)
            current = []
    if current:
        blocks.append(current)
    return blocks

def integer_rows(block):
    return [line.split('\t') for line in block if not line.startswith('#') and line.split('\t', 1)[0].isdigit()]

gold_blocks = read_conllu(TEST_PATH)
pretokenized = [[row[1] for row in integer_rows(block)] for block in gold_blocks]
gold_sentence_count = len(gold_blocks)
gold_word_count = sum(map(len, pretokenized))
assert (gold_sentence_count, gold_word_count) == (100, 1360)
print('gold sentences:', gold_sentence_count, 'gold integer-ID words:', gold_word_count)

def safe_field(value):
    return '_' if value is None or value == '' else str(value)

def write_prediction_from_gold(gold_blocks, doc, output_path):
    if len(doc.sentences) != len(gold_blocks):
        raise RuntimeError(f'句数变化：gold={len(gold_blocks)}, predicted={len(doc.sentences)}')
    output_blocks = []
    strict_correct = 0
    strict_correct_no_punct = 0
    strict_total = 0
    strict_total_no_punct = 0
    for sentence_index, (block, predicted_sentence) in enumerate(zip(gold_blocks, doc.sentences), start=1):
        gold_rows = integer_rows(block)
        predicted_words = predicted_sentence.words
        if len(gold_rows) != len(predicted_words):
            raise RuntimeError(f'第 {sentence_index} 句词数变化：gold={len(gold_rows)}, predicted={len(predicted_words)}')
        predicted_by_id = {}
        for gold_row, predicted_word in zip(gold_rows, predicted_words):
            if gold_row[1] != predicted_word.text:
                raise RuntimeError(f'第 {sentence_index} 句 FORM 不一致：{gold_row[1]!r} != {predicted_word.text!r}')
            predicted_by_id[gold_row[0]] = predicted_word
            correct = int(gold_row[6]) == predicted_word.head and gold_row[7] == predicted_word.deprel
            strict_correct += int(correct)
            strict_total += 1
            if gold_row[3] != 'PUNCT':
                strict_correct_no_punct += int(correct)
                strict_total_no_punct += 1
        rendered = []
        for line in block:
            if line.startswith('#'):
                rendered.append(line)
                continue
            fields = line.split('\t')
            if fields[0].isdigit():
                word = predicted_by_id[fields[0]]
                fields[2] = safe_field(word.lemma)
                fields[3] = safe_field(word.upos)
                fields[4] = safe_field(word.xpos)
                fields[5] = safe_field(word.feats)
                fields[6] = str(word.head)
                fields[7] = safe_field(word.deprel)
                fields[8] = '_'
                line = '\t'.join(fields)
            rendered.append(line)
        output_blocks.append('\n'.join(rendered))
    output_path.write_text('\n\n'.join(output_blocks) + '\n\n', encoding='utf-8')
    return {
        'strict_las_full_deprel': strict_correct / strict_total,
        'strict_las_full_deprel_no_punct': strict_correct_no_punct / strict_total_no_punct,
        'strict_correct': strict_correct,
        'strict_total': strict_total,
    }


In [ ]:
import gc, importlib.util, random, time
import numpy as np
import pandas as pd
from huggingface_hub import scan_cache_dir
from stanza.pipeline.core import DownloadMethod

EVALUATOR_URL = 'https://universaldependencies.org/conll18/conll18_ud_eval.py'
EXPECTED_EVALUATOR_SHA256 = '1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'
EVALUATOR_PATH = Path('/content/conll18_ud_eval.py')
evaluator_bytes = urllib.request.urlopen(EVALUATOR_URL).read()
evaluator_sha256 = hashlib.sha256(evaluator_bytes).hexdigest()
if evaluator_sha256 != EXPECTED_EVALUATOR_SHA256:
    raise RuntimeError(f'官方 evaluator SHA-256 不匹配：{evaluator_sha256}。拒绝使用变化的评分代码。')
EVALUATOR_PATH.write_bytes(evaluator_bytes)
spec = importlib.util.spec_from_file_location('conll18_ud_eval', EVALUATOR_PATH)
conll18_ud_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(conll18_ud_eval)
print('official CoNLL-2018 evaluator SHA-256:', evaluator_sha256)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

gold_ud = conll18_ud_eval.load_conllu_file(str(TEST_PATH))

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def processor_artifacts(config):
    artifacts = {}
    for processor, package_name in config['processors'].items():
        path = MODEL_DIR / config['lang'] / processor / f'{package_name}.pt'
        artifacts[processor] = {
            'package': package_name,
            'path': str(path),
            'sha256': sha256_file(path),
            'registry_md5': resources[config['lang']][processor][package_name]['md5'],
        }
    return artifacts

def cached_hf_revisions(repo_id):
    if repo_id is None:
        return []
    return sorted({
        revision.commit_hash
        for repo in scan_cache_dir().repos if repo.repo_id == repo_id
        for revision in repo.revisions
    })

def run_one_config(config):
    config_id = config['config_id']
    processor_map = config['processors']
    print(f'\n===== {config_id} =====')
    stanza.download(
        lang=config['lang'], model_dir=str(MODEL_DIR), processors=processor_map,
        package=None, logging_level='WARN'
    )
    start = time.time()
    nlp = stanza.Pipeline(
        lang=config['lang'], dir=str(MODEL_DIR), processors=processor_map,
        package=None, tokenize_pretokenized=True,
        download_method=DownloadMethod.REUSE_RESOURCES,
        use_gpu=torch.cuda.is_available(), verbose=False
    )
    doc = nlp(pretokenized)
    prediction_path = RESULT_DIR / f'{config_id}.conllu'
    strict = write_prediction_from_gold(gold_blocks, doc, prediction_path)
    system_ud = conll18_ud_eval.load_conllu_file(str(prediction_path))
    scores = conll18_ud_eval.evaluate(gold_ud, system_ud)
    transformer_model = ('hfl/chinese-electra-180g-large-discriminator'
                         if processor_map['depparse'] == 'gsdsimp_electra-large' else None)
    result = {
        'config_id': config_id,
        'language': config['lang'],
        'language_name': config['lang_name'],
        'package_aliases': ','.join(config['aliases']),
        'tokenize_model': processor_map['tokenize'],
        'pos_model': processor_map['pos'],
        'lemma_model': processor_map['lemma'],
        'depparse_model': processor_map['depparse'],
        'pos_model_registry_md5': resources[config['lang']]['pos'][processor_map['pos']]['md5'],
        'depparse_model_registry_md5': resources[config['lang']]['depparse'][processor_map['depparse']]['md5'],
        'transformer_model': transformer_model,
        'transformer_cached_revisions': cached_hf_revisions(transformer_model),
        'processor_artifacts': processor_artifacts(config),
        'LAS_CoNLL18_percent': 100 * scores['LAS'].f1,
        'UAS_CoNLL18_percent': 100 * scores['UAS'].f1,
        'UPOS_CoNLL18_percent': 100 * scores['UPOS'].f1,
        'LAS_strict_full_deprel_percent': 100 * strict['strict_las_full_deprel'],
        'LAS_strict_full_deprel_no_punct_percent': 100 * strict['strict_las_full_deprel_no_punct'],
        'strict_correct': strict['strict_correct'],
        'strict_total': strict['strict_total'],
        'elapsed_seconds_including_load_and_inference': time.time() - start,
        'prediction_sha256': hashlib.sha256(prediction_path.read_bytes()).hexdigest(),
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

results, failures = [], []
for config in configs:
    try:
        results.append(run_one_config(config))
    except Exception as exc:
        failure = {
            'config_id': config['config_id'],
            'error_type': type(exc).__name__,
            'error': str(exc),
        }
        failures.append(failure)
        print('FAILED but continuing:', json.dumps(failure, ensure_ascii=False, indent=2))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not results:
    raise RuntimeError(f'全部配置均失败：{failures}')
results_df = pd.DataFrame(results).sort_values('LAS_CoNLL18_percent', ascending=False)
display(results_df)
ax = results_df.set_index('config_id')[[
    'LAS_CoNLL18_percent', 'LAS_strict_full_deprel_percent'
]].plot.barh(figsize=(10, 5), title='Zero-shot Stanza Chinese parsers on Cantonese-HK custom test')
ax.set_xlabel('Score (%)')
ax.figure.tight_layout()
ax.figure.savefig(RESULT_DIR / 'las_comparison.png', dpi=180, bbox_inches='tight')


In [ ]:
import platform, shutil

run_manifest = {
    'evaluation_split': 'test',
    'test_was_used_to_compare_model_families': True,
    'evaluation_condition': 'Gold sentence boundaries and gold integer-ID FORM tokenization; Stanza predicts POS, lemma, HEAD and DEPREL.',
    'depparse_pos_input': 'Predicted POS from the configured Stanza POS processor; gold UPOS is used only for evaluation.',
    'primary_metric': 'Official CoNLL-2018 LAS from the pinned UD shared-task evaluator (DEPREL subtypes ignored by that evaluator).',
    'secondary_metrics': ['Strict LAS with complete DEPREL', 'Strict LAS with complete DEPREL excluding gold UPOS=PUNCT'],
    'future_selection_policy': 'After this test comparison, select all LoRA hyperparameters and checkpoints using dev only; do not repeatedly inspect test.',
    'evaluator_url': EVALUATOR_URL,
    'evaluator_sha256': evaluator_sha256,
    'warning': 'These are zero-shot Mandarin-parser baselines on Cantonese, not fine-tuned results and not end-to-end tokenizer scores.',
    'stanza_version': stanza.__version__,
    'stanza_resources_sha256': resources_sha256,
    'test_path': str(TEST_PATH),
    'raw_source_url': RAW_URL,
    'raw_source_sha256': raw_sha256,
    'test_original_positions_1_based': TEST_ORIGINAL_POSITIONS,
    'test_sha256': actual_sha256,
    'sentence_count': gold_sentence_count,
    'word_count': gold_word_count,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'python': platform.python_version(),
    'torch': torch.__version__,
    'configs': configs,
    'results': results,
    'failures': failures,
}
(RESULT_DIR / 'results.json').write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
results_df.to_csv(RESULT_DIR / 'results.csv', index=False)
archive = shutil.make_archive('/content/yue_test_stanza_zh_results', 'zip', root_dir=RESULT_DIR)
print('结果目录:', RESULT_DIR)
print('压缩包:', archive)
files.download(archive)
